# Prosperity 4 Tutorial Round Explorer

This notebook gives a practical first-pass dashboard for `EMERALDS` and `TOMATOES`:
- Price dynamics (bid/mid/ask)
- Spread and liquidity behavior
- Order-book imbalance and volatility
- Trade flow, quantity, and notional patterns

In [1]:
from pathlib import Path
import re
import sys
from typing import cast

import pandas as pd
import plotly.express as px

repo_root = Path.cwd().resolve().parents[1] if Path.cwd().name == 'analysis' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.analysis.data import read_all_tutorial_prices

prices = read_all_tutorial_prices()

tutorial_dir = repo_root / 'data' / 'tutorial_round'
trade_frames: list[pd.DataFrame] = []
for path in sorted(tutorial_dir.glob('trades_round_0_day_*.csv')):
    day_match = re.search(r'day_(-?\d+)$', path.stem)
    day = int(day_match.group(1)) if day_match else 0
    frame = cast(pd.DataFrame, pd.read_csv(filepath_or_buffer=str(path), sep=';'))
    frame['day'] = day
    trade_frames.append(frame)

trades = pd.concat(trade_frames, ignore_index=True) if trade_frames else pd.DataFrame()

print(f'Prices shape: {prices.shape}')
print(f'Trades shape: {trades.shape}')
prices.head()

Prices shape: (40000, 17)
Trades shape: (1219, 8)


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,-1,0,TOMATOES,4999,5,4998,15,NaN,NaN,5013,5,5014,15,NaN,NaN,5006.0,0.0
1,-1,0,EMERALDS,9992,14,9990,29,NaN,NaN,10008,14,10010,29,NaN,NaN,10000.0,0.0
2,-1,100,EMERALDS,9992,11,9990,22,NaN,NaN,10008,11,10010,22,NaN,NaN,10000.0,0.0
3,-1,100,TOMATOES,5000,8,4998,21,NaN,NaN,5013,8,5014,21,NaN,NaN,5006.5,0.0
4,-1,200,EMERALDS,9992,15,9990,20,NaN,NaN,10008,15,10010,20,NaN,NaN,10000.0,0.0


In [2]:
df = prices.copy()
df = df.sort_values(['product', 'day', 'timestamp']).reset_index(drop=True)

df['time_index'] = df['day'] * 1_000_000 + df['timestamp']
df['spread'] = df['ask_price_1'] - df['bid_price_1']
df['mid'] = (df['ask_price_1'] + df['bid_price_1']) / 2

bid_v = df['bid_volume_1'].abs()
ask_v = df['ask_volume_1'].abs()
top_depth = bid_v + ask_v

df['imbalance'] = (bid_v - ask_v).where(top_depth > 0, 0) / top_depth.where(top_depth > 0, 1)
df['microprice'] = ((df['ask_price_1'] * bid_v) + (df['bid_price_1'] * ask_v)).where(top_depth > 0, df['mid']) / top_depth.where(top_depth > 0, 1)
df['mid_return'] = df.groupby('product')['mid'].pct_change().fillna(0.0)
df['rolling_vol_100'] = (
    df.groupby('product')['mid_return']
    .rolling(100)
    .std()
    .reset_index(level=0, drop=True)
)

trades = trades.rename(columns={'symbol': 'product'})
trades['time_index'] = trades['day'] * 1_000_000 + trades['timestamp']
trades['notional'] = trades['price'] * trades['quantity']

spread_stats = (
    df.groupby('product')['spread']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .sort_index()
)
spread_stats

,count,mean,median,std,min,max
product,,,,,,
EMERALDS,20000,15.73840,16.0,1.422838,8,16
TOMATOES,20000,13.02025,13.0,1.754592,5,14


In [3]:
liquidity_stats = (
    df.groupby('product')
    .agg(
        mean_spread=('spread', 'mean'),
        mean_depth=('bid_volume_1', lambda s: s.abs().mean()),
        mean_ask_depth=('ask_volume_1', lambda s: s.abs().mean()),
        mean_abs_imbalance=('imbalance', lambda s: s.abs().mean()),
    )
    .sort_index()
)
liquidity_stats

,mean_spread,mean_depth,mean_ask_depth,mean_abs_imbalance
product,,,,
EMERALDS,15.73840,12.45705,12.45980,0.007996
TOMATOES,13.02025,7.44100,7.44505,0.018787


In [4]:
sample = (
    df.groupby('product', as_index=False, group_keys=False)
    .head(3000)
    .copy()
)

price_panel = sample.melt(
    id_vars=['product', 'time_index'],
    value_vars=['bid_price_1', 'mid', 'ask_price_1'],
    var_name='series',
    value_name='price',
)

fig_price = px.line(
    price_panel,
    x='time_index',
    y='price',
    color='series',
    facet_row='product',
    title='Top-of-Book Prices (first 3000 points per product)',
)
fig_price.update_layout(height=700)
fig_price.show()

In [5]:
fig_spread = px.line(
    sample,
    x='time_index',
    y='spread',
    color='product',
    title='Bid/Ask Spread Over Time (first 3000 points per product)',
)
fig_spread.show()

In [6]:
fig_spread_dist = px.box(
    df,
    x='product',
    y='spread',
    color='product',
    points=False,
    title='Spread Distribution by Product',
)
fig_spread_dist.show()

In [7]:
fig_imbalance = px.line(
    sample,
    x='time_index',
    y='imbalance',
    color='product',
    title='Top-of-Book Imbalance Over Time (first 3000 points per product)',
)
fig_imbalance.show()

In [8]:
vol_sample = df.groupby('product', as_index=False, group_keys=False).head(4000)
fig_vol = px.line(
    vol_sample,
    x='time_index',
    y='rolling_vol_100',
    color='product',
    title='Rolling Mid-Return Volatility (window=100)',
)
fig_vol.show()

In [9]:
trade_activity = (
    trades.groupby(['product', 'time_index'], as_index=False)
    .agg(quantity=('quantity', 'sum'), notional=('notional', 'sum'))
)

fig_qty = px.line(
    trade_activity,
    x='time_index',
    y='quantity',
    color='product',
    title='Executed Quantity Over Time',
)
fig_qty.show()

fig_notional = px.line(
    trade_activity,
    x='time_index',
    y='notional',
    color='product',
    title='Executed Notional Over Time',
)
fig_notional.show()

In [10]:
fig_trade_price = px.histogram(
    trades,
    x='price',
    color='product',
    barmode='overlay',
    nbins=60,
    title='Trade Price Distribution',
)
fig_trade_price.show()

In [11]:
signal_view = df[['product', 'imbalance', 'mid_return']].copy()
fig_signal = px.scatter(
    signal_view.sample(min(len(signal_view), 8000), random_state=7),
    x='imbalance',
    y='mid_return',
    color='product',
    opacity=0.35,
    title='Imbalance vs Next Mid Return Proxy (sanity-check scatter)',
)
fig_signal.show()
